# External tumor-only mask preprocessing

**Scientific purpose.** Select tumor segments from HCC-TACE-Seg DICOM-SEG objects, map
their frames to the referenced converted CT geometry, and construct a private external
inference manifest.

**Runnability classification:** requires private DICOM, converted CT geometry, and an
operator-controlled review/resume step after notebook 08. **Inputs:** private DICOM-SEG and
converted CT geometry.
**Private outputs:** segment decisions, tumor-only
NIfTI masks, mapping diagnostics, and a patient-linked manifest.

Segment labels that are ambiguous must be reviewed; organ masks are never silently used
as tumor masks. No patient-level mask or overlay is released.


In [ ]:
from pathlib import Path
import sys


def locate_repository(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = locate_repository()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from liver_tumor_pipeline.config import load_config, require_path

CONFIG_PATH = REPO_ROOT / "configs" / "paths.yaml"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(
        "Create an untracked configs/paths.yaml from configs/paths.example.yaml and "
        "set the documented environment variables before running this workflow."
    )

CONFIG = load_config(CONFIG_PATH)
PRIVATE_ARTIFACT_ROOT = require_path(CONFIG, "private_artifact_root", must_exist=False)
OUTPUT_ROOT = require_path(CONFIG, "output_root", must_exist=False)

import json
import re
from collections.abc import Iterable

import numpy as np
import pandas as pd
import pydicom
import SimpleITK as sitk

EXTERNAL_WORK_ROOT = PRIVATE_ARTIFACT_ROOT / "external"
CONVERSION_ROOT = EXTERNAL_WORK_ROOT / "ct_conversion"
CATALOG_PATH = CONVERSION_ROOT / "dicom_catalog.csv"
CT_CONVERSION_PATH = CONVERSION_ROOT / "ct_conversion.csv"
MASK_ROOT = EXTERNAL_WORK_ROOT / "tumor_masks"
MASK_NIFTI_ROOT = MASK_ROOT / "nifti"
SEGMENT_REVIEW_PATH = MASK_ROOT / "segment_review.csv"
MASK_CONVERSION_PATH = MASK_ROOT / "mask_conversion.csv"
MANIFEST_PATH = MASK_ROOT / "external_manifest.csv"
for required in (CATALOG_PATH, CT_CONVERSION_PATH):
    if not required.is_file():
        raise FileNotFoundError("Private DICOM catalog or CT conversion manifest is unavailable.")
MASK_NIFTI_ROOT.mkdir(parents=True, exist_ok=True)
catalog = pd.read_csv(CATALOG_PATH)
ct_conversion = pd.read_csv(CT_CONVERSION_PATH)
PHASE_ORDER = ("P", "C1", "C2", "C3")


## Tumor-segment decisions

Tumor-related labels are included and known organ/background labels are excluded.
Remaining labels are marked for manual review and block conversion until resolved in the
private decision table.


In [ ]:
TUMOR_TERMS = (
    "tumor", "tumour", "lesion", "hcc", "target", "mass", "neoplasm",
    "carcinoma", "malignant", "viable", "necrosis", "treated", "tace",
)
EXCLUDED_TERMS = (
    "liver", "hepatic", "parenchyma", "organ", "background", "vessel", "vein",
    "artery", "aorta", "cava", "spleen", "kidney", "gallbladder", "bowel", "bone",
)


def normalize_label(*values) -> str:
    return re.sub(r"\s+", " ", " ".join(str(value) for value in values).lower()).strip()


def segment_decisions(seg_path: Path, patient_key: str, series_uid: str) -> list[dict]:
    dataset = pydicom.dcmread(str(seg_path), force=True)
    rows = []
    for segment in getattr(dataset, "SegmentSequence", []):
        label = str(getattr(segment, "SegmentLabel", ""))
        description = str(getattr(segment, "SegmentDescription", ""))
        code_meaning = ""
        sequence = getattr(segment, "SegmentedPropertyTypeCodeSequence", [])
        if sequence:
            code_meaning = str(getattr(sequence[0], "CodeMeaning", ""))
        text = normalize_label(label, description, code_meaning)
        tumor_hit = any(term in text for term in TUMOR_TERMS)
        excluded_hit = any(term in text for term in EXCLUDED_TERMS)
        if tumor_hit and not excluded_hit:
            decision = "include_tumor"
        elif excluded_hit and not tumor_hit:
            decision = "exclude"
        elif tumor_hit and any(term in text for term in ("tumor", "lesion", "hcc", "mass")):
            decision = "include_tumor"
        else:
            decision = "manual_review"
        rows.append(
            {
                "patient_key": patient_key,
                "seg_series_uid": series_uid,
                "segment_number": int(segment.SegmentNumber),
                "segment_label": label,
                "segment_description": description,
                "code_meaning": code_meaning,
                "decision": decision,
            }
        )
    return rows


def first_seg_file(group: pd.DataFrame) -> Path:
    files = [Path(path) for path in group["file_path"].astype(str)]
    if not files:
        raise FileNotFoundError("DICOM-SEG series has no indexed file.")
    return files[0]


review_rows = []
seg_catalog = catalog[catalog["modality"] == "SEG"]
for (patient_key, series_uid), group in seg_catalog.groupby(["patient_key", "series_uid"]):
    review_rows.extend(segment_decisions(first_seg_file(group), str(patient_key), str(series_uid)))
segment_review = pd.DataFrame(review_rows)
segment_review.to_csv(SEGMENT_REVIEW_PATH, index=False)
if segment_review.empty:
    raise RuntimeError("No DICOM-SEG segment definitions were discovered.")
if (segment_review["decision"] == "manual_review").any():
    raise RuntimeError(
        "Ambiguous DICOM-SEG labels require documented private review before conversion."
    )
segment_review["decision"].value_counts().rename_axis("decision").reset_index(name="segments")


## Reference-aware frame mapping

Frames are filtered by selected segment number and placed on the referenced CT slice by
SOP Instance UID, with nearest Image Position Patient as a documented fallback. The mask
inherits the converted CT geometry.


In [ ]:
def referenced_sop_uids(value) -> set[str]:
    references: set[str] = set()
    def walk(item) -> None:
        if isinstance(item, pydicom.dataset.Dataset):
            if hasattr(item, "ReferencedSOPInstanceUID"):
                references.add(str(item.ReferencedSOPInstanceUID))
            for element in item:
                if element.VR == "SQ":
                    for child in element.value:
                        walk(child)
        elif isinstance(item, Iterable) and not isinstance(item, (str, bytes)):
            for child in item:
                walk(child)
    walk(value)
    return references


def frame_segment_number(functional_group) -> int | None:
    sequence = getattr(functional_group, "SegmentIdentificationSequence", [])
    return int(sequence[0].ReferencedSegmentNumber) if sequence else None


def choose_referenced_ct(patient_key: str, seg_dataset) -> pd.Series:
    candidates = ct_conversion[ct_conversion["patient_key"].astype(str) == patient_key].copy()
    if candidates.empty:
        raise ValueError("No converted CT series is available for a DICOM-SEG patient.")
    references = referenced_sop_uids(seg_dataset)
    if references:
        candidates["reference_overlap"] = candidates["ordered_sop_uids"].map(
            lambda value: len(references.intersection(json.loads(value)))
        )
        candidates = candidates.sort_values("reference_overlap", ascending=False)
        if int(candidates.iloc[0]["reference_overlap"]) > 0:
            return candidates.iloc[0]
    phase_rank = {"C2": 0, "C1": 1, "C3": 2, "P": 3, "unknown": 4}
    candidates["phase_rank"] = candidates["phase"].map(phase_rank).fillna(5)
    return candidates.sort_values("phase_rank").iloc[0]


def selected_segment_numbers(seg_series_uid: str) -> set[int]:
    rows = segment_review[
        (segment_review["seg_series_uid"].astype(str) == seg_series_uid)
        & (segment_review["decision"] == "include_tumor")
    ]
    return {int(value) for value in rows["segment_number"]}


def convert_tumor_seg(seg_path: Path, patient_key: str, seg_series_uid: str) -> dict:
    dataset = pydicom.dcmread(str(seg_path), force=True)
    selected = selected_segment_numbers(seg_series_uid)
    if not selected:
        raise ValueError("DICOM-SEG series has no reviewed tumor segment.")
    ct_row = choose_referenced_ct(patient_key, dataset)
    ct_image = sitk.ReadImage(str(ct_row["ct_path"]))
    ct_array = sitk.GetArrayFromImage(ct_image)
    mask = np.zeros_like(ct_array, dtype=np.uint8)
    pixels = dataset.pixel_array
    if pixels.ndim == 2:
        pixels = pixels[None]
    functional_groups = getattr(dataset, "PerFrameFunctionalGroupsSequence", None)
    if functional_groups is None:
        raise ValueError("SEG lacks per-frame geometry required for safe segment selection.")
    sop_to_slice = {
        sop_uid: index for index, sop_uid in enumerate(json.loads(ct_row["ordered_sop_uids"]))
    }
    position_rows = json.loads(ct_row["ordered_positions"])
    position_to_slice = [
        (np.asarray(position, dtype=float), index)
        for index, position in enumerate(position_rows)
        if position is not None
    ]
    mapped_frames = 0
    for index, functional_group in enumerate(functional_groups):
        if frame_segment_number(functional_group) not in selected:
            continue
        frame = (pixels[index] > 0).astype(np.uint8)
        if frame.sum() == 0:
            continue
        slice_index = None
        for sop_uid in referenced_sop_uids(functional_group):
            if sop_uid in sop_to_slice:
                slice_index = sop_to_slice[sop_uid]
                break
        plane_positions = getattr(functional_group, "PlanePositionSequence", [])
        if slice_index is None and plane_positions and position_to_slice:
            position = np.asarray(plane_positions[0].ImagePositionPatient, dtype=float)
            slice_index = min(
                position_to_slice,
                key=lambda item: float(np.linalg.norm(position - item[0])),
            )[1]
        if slice_index is None:
            continue
        if frame.shape != mask[slice_index].shape:
            raise ValueError("DICOM-SEG frame and referenced CT slice shapes differ.")
        mask[slice_index] = np.maximum(mask[slice_index], frame)
        mapped_frames += 1
    if mask.sum() == 0 or mapped_frames == 0:
        raise RuntimeError("Tumor-only SEG conversion produced an empty mask.")
    output_path = MASK_NIFTI_ROOT / f"{patient_key}__{seg_series_uid}__tumor.nii.gz"
    mask_image = sitk.GetImageFromArray(mask)
    mask_image.CopyInformation(ct_image)
    sitk.WriteImage(mask_image, str(output_path), useCompression=True)
    return {
        "patient_key": patient_key,
        "seg_series_uid": seg_series_uid,
        "referenced_ct_series_uid": str(ct_row["series_uid"]),
        "referenced_ct_path": str(ct_row["ct_path"]),
        "tumor_mask_path": str(output_path),
        "selected_segments": json.dumps(sorted(selected)),
        "mapped_frames": mapped_frames,
        "tumor_voxels": int(mask.sum()),
    }


In [ ]:
def convert_all_tumor_masks() -> pd.DataFrame:
    records = []
    for (patient_key, series_uid), group in seg_catalog.groupby(["patient_key", "series_uid"]):
        records.append(
            convert_tumor_seg(first_seg_file(group), str(patient_key), str(series_uid))
        )
    converted = pd.DataFrame(records)
    if converted.empty:
        raise RuntimeError("No tumor-only masks were converted.")
    converted.to_csv(MASK_CONVERSION_PATH, index=False)
    return converted


def series_depth(value: str) -> int:
    shape = json.loads(value)
    return int(shape[2]) if len(shape) == 3 else 0


def choose_phase_candidate(candidates: pd.DataFrame, phase: str) -> pd.Series | None:
    phase_rows = candidates[candidates["phase"] == phase].copy()
    if phase_rows.empty:
        return None
    phase_rows["depth"] = phase_rows["shape_xyz"].map(series_depth)
    return phase_rows.sort_values("depth", ascending=False).iloc[0]


def build_external_manifest(mask_table: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for patient_key, ct_rows in ct_conversion.groupby("patient_key"):
        patient_masks = mask_table[mask_table["patient_key"].astype(str) == str(patient_key)]
        if patient_masks.empty:
            continue
        primary_mask = patient_masks.iloc[0]
        referenced = ct_rows[
            ct_rows["series_uid"].astype(str)
            == str(primary_mask["referenced_ct_series_uid"])
        ]
        primary_ct = referenced.iloc[0] if not referenced.empty else ct_rows.iloc[0]
        phase_paths = {}
        for phase in PHASE_ORDER:
            selected = choose_phase_candidate(ct_rows, phase)
            phase_paths[phase] = "" if selected is None else str(selected["ct_path"])
        c2_fallback = not bool(phase_paths["C2"])
        if c2_fallback:
            phase_paths["C2"] = str(primary_ct["ct_path"])
        mask_path = str(primary_mask["tumor_mask_path"])
        rows.append(
            {
                "patient_key": str(patient_key),
                "label": "HCC",
                "ct_P": phase_paths["P"],
                "ct_C1": phase_paths["C1"],
                "ct_C2": phase_paths["C2"],
                "ct_C3": phase_paths["C3"],
                "mask_P": mask_path if phase_paths["P"] else "",
                "mask_C1": mask_path if phase_paths["C1"] else "",
                "mask_C2": mask_path,
                "mask_C3": mask_path if phase_paths["C3"] else "",
                "c2_fallback_used": c2_fallback,
                "real_phase_count": int(sum(bool(phase_paths[p]) for p in PHASE_ORDER))
                - int(c2_fallback),
            }
        )
    manifest = pd.DataFrame(rows)
    if manifest.empty:
        raise RuntimeError("No external patient has both converted CT and tumor mask data.")
    manifest.to_csv(MANIFEST_PATH, index=False)
    return manifest


## Private conversion and manifest boundary

After reviewing the private segment table, call `convert_all_tumor_masks()`, then provide
its returned table to `build_external_manifest()`. The historical
adapter used a primary CT as a C2 fallback when C2 was absent, but did not claim that
duplicated or missing phase slots were true acquisitions. No patient had all four required
phases. The external cohort is known-HCC only and cannot support external multiclass AUC.

The released decision-generation cell writes `SEGMENT_REVIEW_PATH` and stops when manual
review is required. Complete that review in private storage, then reload the reviewed table
into `segment_review` before calling the conversion helpers. This operator-controlled resume
step is not automated in the public notebook; rerunning the generation cell would replace the
review file.
